# 04 · 原片精改 v2 专业高清管线（IndexTTS-2 播音级清晰中文克隆 + 时序平滑 FFHQ-512 换脸/BiSeNet 遮罩 + ByteDance LatentSync 1.6 512px 口型）

- **针对性根治 v1 两大痛点**：
  1. **台词彻底清晰**：改用 Bilibili **IndexTTS-2**（字符+拼音混合建模 + BigVGAN2 声码器），严格保持 **`1.00x` 原生清晰语速**（废除 0.82x 慢放变形），并将说话段原片 Demucs 残声压低 `-18dB`，消除重影。
  2. **人物彻底清晰稳定**：新增 **5 点关键点跨帧 EMA 时序平滑**（防抖）+ **标准 5 点 FFHQ 512×512 仿射对齐 GFPGANv1.4 还原**（修复未对齐发糊）+ **BiSeNet 像素级语义遮罩**（保护清宫帽檐与胡须）+ **ByteDance LatentSync 1.6 (`512×512` 扩散高清口型)**（28 倍于 96×96 Wav2Lip 像素面积）。
  3. **15 秒快速验收**：默认 `CLIP_END_SEC = 17.2`（完整覆盖特朗普 `shot05c` + 鲁比奥 `shot06` 两句对白），L4 GPU 约 4–6 分钟跑完并自动释放算力。

In [ ]:
#@title 1. 挂载 5TB Google Drive、拉取最新仓库并构建独立 GPU 环境（IndexTTS-2 + 换脸修复 + LatentSync 1.6）
REPO_URL = "https://github.com/WSQ233/AI-Video.git" #@param {type:"string"}
REPO_DIR = "/content/AI-Video" #@param {type:"string"}
PROJECT_ID = "02_yongzheng_ep12_huang" #@param {type:"string"}

import os, shutil, subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

def run_stream(cmd, cwd=None, extra_env=None):
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed ({rc}): {cmd}")

if not Path(REPO_DIR).exists():
    run_stream(["git", "clone", REPO_URL, REPO_DIR])
else:
    run_stream(["git", "-C", REPO_DIR, "pull", "--ff-only"])

PROJECT_DIR = Path(f"/content/drive/MyDrive/AI_Video/projects/{PROJECT_ID}")
shutil.copy2(f"{REPO_DIR}/projects/{PROJECT_ID}/dubbing_plan_shot01_21.json", PROJECT_DIR / "dubbing_plan_shot01_21.json")

run_stream(["nvidia-smi"])
run_stream(["pip", "install", "-q", "-U", "uv", "huggingface_hub"])

# 环境 A：Bilibili IndexTTS-2（原生 uv 隔离环境 + 5TB Drive 权重缓存）
INDEXTTS_DIR = Path("/content/index-tts")
if not INDEXTTS_DIR.exists():
    run_stream(["git", "clone", "--depth", "1", "https://github.com/index-tts/index-tts.git", str(INDEXTTS_DIR)])
run_stream(["uv", "sync", "--no-dev"], cwd=str(INDEXTTS_DIR))
run_stream(["uv", "pip", "install", "--python", str(INDEXTTS_DIR / ".venv/bin/python"), "demucs", "soundfile"])

CKPT_TTS = Path("/content/drive/MyDrive/AI_Video/shared/models/indextts2")
CKPT_TTS.mkdir(parents=True, exist_ok=True)
if not (CKPT_TTS / "config.yaml").exists():
    print("[下载权重] IndexTeam/IndexTTS-2 -> 5TB Google Drive 共享缓存...", flush=True)
    run_stream([
        "python3", "-c",
        f"from huggingface_hub import snapshot_download; snapshot_download('IndexTeam/IndexTTS-2', local_dir='{CKPT_TTS}')"
    ])

# 环境 B：Python 3.10 视觉 + LatentSync 1.6 统一 CUDA 环境
VIS_VENV = Path("/content/venv_vis")
if not VIS_VENV.exists():
    run_stream(["uv", "venv", "--python", "3.10", str(VIS_VENV)])
VIS_PY = str(VIS_VENV / "bin/python")

LS_DIR = Path("/content/LatentSync")
if not LS_DIR.exists():
    run_stream(["git", "clone", "--depth", "1", "https://github.com/bytedance/LatentSync.git", str(LS_DIR)])

run_stream([
    "uv", "pip", "install", "--python", VIS_PY,
    "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu124"
])
run_stream([
    "uv", "pip", "install", "--python", VIS_PY,
    "diffusers==0.32.2", "transformers==4.48.0", "accelerate==0.26.1", "decord==0.6.0",
    "einops==0.7.0", "omegaconf==2.3.0", "kornia==0.8.0", "imageio==2.36.1", "imageio-ffmpeg==0.5.1",
    "mediapipe==0.10.14", "face-alignment==1.4.1", "scenedetect", "ffmpeg-python", "av", "lpips", "python-speech-features", "huggingface_hub",
    "insightface==0.7.3", "gfpgan==1.3.8", "basicsr==1.4.2", "facexlib==0.3.0",
    "opencv-python-headless", "scipy", "scikit-learn", "pillow", "soundfile", "librosa"
])
run_stream(["uv", "pip", "install", "--python", VIS_PY, "--force-reinstall", "onnxruntime-gpu"])

# 修补 basicsr 与 torchvision>=0.18 兼容性
run_stream([
    VIS_PY, "-c",
    "import pathlib; p=pathlib.Path('/content/venv_vis/lib/python3.10/site-packages/basicsr/data/degradations.py'); "
    "t=p.read_text(); p.write_text(t.replace('from torchvision.transforms.functional_tensor import rgb_to_grayscale', 'from torchvision.transforms.functional import rgb_to_grayscale'))"
])

# 缓存 ByteDance/LatentSync-1.6 (512x512) 权重至 5TB Drive
CKPT_LS = Path("/content/drive/MyDrive/AI_Video/shared/models/latentsync16")
CKPT_LS.mkdir(parents=True, exist_ok=True)
if not (CKPT_LS / "latentsync_unet.pt").exists():
    print("[下载权重] ByteDance/LatentSync-1.6 (512x512) -> 5TB Google Drive 共享缓存...", flush=True)
    run_stream([
        "python3", "-c",
        f"from huggingface_hub import snapshot_download; snapshot_download('ByteDance/LatentSync-1.6', local_dir='{CKPT_LS}')"
    ])
if not (CKPT_LS / "whisper" / "tiny.pt").exists():
    run_stream([
        "python3", "-c",
        f"from huggingface_hub import hf_hub_download; hf_hub_download('ByteDance/LatentSync-1.6', 'whisper/tiny.pt', local_dir='{CKPT_LS}')"
    ])

print("\n[v2 环境全部就绪] IndexTTS-2 + FFHQ-512 换脸/BiSeNet + ByteDance LatentSync 1.6 就绪！")

In [ ]:
#@title 2. Stage 1：精确裁剪 + IndexTTS-2 播音级清晰中文克隆（1.00x 原速，零重影）
WORK_DIR = "/content/work_replace_v2" #@param {type:"string"}
CLIP_END_SEC = 17.2 #@param {type:"number"}

import glob, os
from IPython.display import Audio, display

TTS_PY = "/content/index-tts/.venv/bin/python"
nv_libs = glob.glob("/content/index-tts/.venv/lib/python*/site-packages/nvidia/*/lib")
run_stream([
    TTS_PY, "-u", f"{REPO_DIR}/scripts/video_replace_v2.py",
    "--project-dir", str(PROJECT_DIR),
    "--work-dir", WORK_DIR,
    "--stage", "dub",
    "--clip-end-sec", str(CLIP_END_SEC),
    "--indextts-dir", "/content/index-tts",
    "--indextts-ckpt-dir", str(CKPT_TTS),
], extra_env={"LD_LIBRARY_PATH": ":".join(nv_libs + [os.environ.get("LD_LIBRARY_PATH", "")])})

for wav in sorted(Path(WORK_DIR, "tts_v2").glob("*_clean.wav")):
    print(f"试听 IndexTTS-2 清晰中文克隆: {wav.name}")
    display(Audio(str(wav)))

In [ ]:
#@title 3. Stage 2 & 3：时序平滑 FFHQ-512 高清换脸/BiSeNet 遮罩 + LatentSync 1.6 (512px) 口型同步 + 自动释放 GPU
OUTPUT_NAME = "02_yongzheng_ep12_huang_v2_15s_preview.mp4" #@param {type:"string"}
INFERENCE_STEPS = 20 #@param {type:"integer"}
SKIP_LIPSYNC = False #@param {type:"boolean"}
AUTO_DISCONNECT = True #@param {type:"boolean"}

import glob, os, subprocess, time
from pathlib import Path

vis_nv_libs = glob.glob("/content/venv_vis/lib/python3.10/site-packages/nvidia/*/lib")
extra_env = {"LD_LIBRARY_PATH": ":".join(vis_nv_libs + [os.environ.get("LD_LIBRARY_PATH", "")])}

# Stage 2: 5点 EMA 时序平滑 + FFHQ-512 对齐 GFPGAN + BiSeNet 语义遮罩
run_stream([
    VIS_PY, "-u", f"{REPO_DIR}/scripts/video_replace_v2.py",
    "--project-dir", str(PROJECT_DIR),
    "--work-dir", WORK_DIR,
    "--stage", "swap",
], extra_env=extra_env)

# Stage 3: ByteDance LatentSync 1.6 (512x512 HD) 说话人精准口型同步 + 终混成片
ls_cmd = [
    VIS_PY, "-u", f"{REPO_DIR}/scripts/video_replace_v2.py",
    "--project-dir", str(PROJECT_DIR),
    "--work-dir", WORK_DIR,
    "--stage", "lipsync",
    "--latentsync-dir", "/content/LatentSync",
    "--latentsync-ckpt-dir", str(CKPT_LS),
    "--output-name", OUTPUT_NAME,
    "--inference-steps", str(INFERENCE_STEPS),
]
if SKIP_LIPSYNC:
    ls_cmd.append("--skip-lipsync")
run_stream(ls_cmd, extra_env=extra_env)

subprocess.run(["sync"], check=False)
time.sleep(10)

final_mp4 = PROJECT_DIR / "outputs" / "final_cut" / OUTPUT_NAME
print(f"\n=== v2 高清验证样片完成！位置: {final_mp4} ===")

if AUTO_DISCONNECT and final_mp4.exists() and final_mp4.stat().st_size > 500_000:
    print("[自动止损] 成片已安全落盘 5TB Google Drive，自动释放 GPU 运行时！")
    from google.colab import runtime
    runtime.unassign()